# Module 0.1: Linux Fundamentals for Bioinformatics

---

### Learning Objectives

By the end of this module, you will be able to:
- Navigate the Linux file system with confidence
- Create, move, copy, and delete files and directories
- Use pipes and redirections to build data-processing pipelines
- Search files with `grep`, `find`, and regular expressions
- Process structured text with `awk` and `sed`
- Work with compressed files (gzip, tar) -- the standard in genomics
- Manage processes and connect to remote servers
- Apply all of the above to real bioinformatics file formats (FASTA, FASTQ, BED, VCF, BAM)

**Prerequisites:** None. This module starts from scratch.

**Estimated time:** 3-4 hours

---

## Why this notebook matters

The Linux command line is the working environment for virtually all bioinformatics tools. STAR, BWA, GATK, samtools, bcftools — none of them have GUIs. HPC clusters run Linux and are accessed only through SSH. Genomic files are gigabytes to terabytes; the only practical way to work with them is at the command line. This notebook teaches the specific subset of Linux you will use every day: navigation, file operations, pipes, grep, awk, sed, and compression.


## How to use this notebook
1. Run cells top-to-bottom — later cells depend on earlier ones (e.g., files created in one cell are used in the next).
2. For every `%%bash` cell: read the command first, understand what it does, then run it.
3. **Experiment**: modify commands and re-run. Breaking things is the best way to learn Linux.
4. If a cell fails, check that previous cells were run and that the working directory is what you expect.


## Common stumbling points

- **Spaces around `=` in Bash**: `var = value` is wrong; `var=value` is right. The space makes Bash interpret `var` as a command name.
- **Relative vs. absolute paths**: `cat data/file.txt` works only from the right directory; `cat /home/user/project/data/file.txt` works from anywhere.
- **`rm` is permanent**: Linux has no trash bin. Always double-check before running `rm -rf`.
- **Pipes discard stderr**: `command1 | command2` only passes stdout. Add `2>&1` to also pipe error messages.
- **Wildcards expand before the command runs**: `rm *.fastq` becomes `rm sample1.fastq sample2.fastq ...` — the shell does the expansion, not `rm`.


---

## 1. Getting Help

Before we start learning commands, the most important skill: **how to look things up.**

Every standard Linux command comes with a manual page (`man`) and usually a `--help` flag.

In [ ]:
%%bash
# The man command opens the manual page for any command
# (Press 'q' to quit, '/' to search within the manual)
# man ls

# Quick help -- most commands support --help
ls --help 2>&1 | head -20

# whatis gives a one-line description
whatis grep
whatis awk

---

## 2. Navigation: Where Am I, What Is Here?

The Linux file system is a tree rooted at `/`. Everything -- devices, processes, your
home directory -- is a file or directory somewhere in this tree.

```
                     / (root)
                     |
     +---------------+----------------+
     |               |                |
    home            bin              usr
     |                                |
     +-- username                    local/bin
          |
          +-- projects/
          |     +-- rna_seq_analysis/
          |     +-- genome_assembly/
          +-- data/
                +-- raw_reads/
                +-- references/
```

### Key path concepts

| Symbol | Meaning | Example |
|--------|---------|---------|
| `/` | Root directory (top of the tree) | `cd /` |
| `~` | Your home directory | `cd ~` (same as `cd /home/username`) |
| `.` | Current directory | `./run_script.sh` |
| `..` | Parent directory (one level up) | `cd ..` |
| `-` | Previous directory (where you just were) | `cd -` |

In [ ]:
%%bash
# pwd -- Print Working Directory: where am I right now?
pwd

# ls -- List directory contents
ls              # Basic listing
ls -l           # Long format: permissions, owner, size, date
ls -a           # Include hidden files (starting with .)
ls -lah         # Long + all + human-readable sizes -- the combo you will use most
ls -1           # One file per line -- useful for piping

In [ ]:
%%bash
# cd -- Change Directory
cd ~              # Go to home directory
cd ..             # Go up one level
cd ../..          # Go up two levels
cd /usr/local/bin # Absolute path (starts with /)
cd projects       # Relative path (from current directory)
cd -              # Go back to the previous directory

# Tip: Press Tab to autocomplete paths -- saves enormous time

---

## 3. File and Directory Operations

### Creating

In [ ]:
%%bash
# mkdir -- Make Directory
mkdir my_project                    # Create a single directory
mkdir -p project/{data/{raw,processed},results,scripts,logs}
#   -p creates parent directories as needed AND does not error if they exist
#   Brace expansion {a,b} creates multiple siblings at once

# touch -- Create an empty file (or update timestamp of existing file)
touch README.md
touch data/sample_{01,02,03}.txt    # Creates three files via brace expansion

### Copying, Moving, Renaming, Deleting

In [ ]:
%%bash
# cp -- Copy
cp source.txt destination.txt       # Copy a file
cp -r source_dir/ dest_dir/         # Copy a directory recursively (-r is required)

# mv -- Move or Rename (same command for both)
mv old_name.txt new_name.txt        # Rename a file
mv file.txt target_directory/       # Move a file into a directory

# rm -- Remove (WARNING: there is no trash bin -- deletion is permanent)
rm file.txt                         # Delete a file
rm -r directory/                    # Delete a directory and everything inside
rm -rf directory/                   # Force-delete without confirmation
#   NEVER run: rm -rf /   or   rm -rf ~   -- you will destroy your system/data

### Bioinformatics context: Project directory conventions

Most bioinformatics projects follow a numbered-folder convention to make the
analysis order obvious:

```bash
mkdir -p RNA_Seq_Project/{00_raw_data,01_qc,02_trimmed,03_aligned,04_counts,05_analysis,scripts,logs}
```

This mirrors the actual pipeline steps: raw reads go in `00_raw_data`, quality
control reports in `01_qc`, and so on. Anyone opening the project folder immediately
sees the workflow.

---

## 4. Viewing File Contents

You will spend a huge amount of time inspecting files -- checking whether a FASTQ file
downloaded correctly, previewing a VCF, or reviewing pipeline logs.

In [ ]:
%%bash
# cat -- Print entire file to terminal (fine for small files)
cat small_file.txt

# less -- Interactive viewer with scrolling (press q to quit, / to search)
# less large_file.txt

# head / tail -- View the beginning or end of a file
head -n 10 file.txt         # First 10 lines
tail -n 10 file.txt         # Last 10 lines
head -n 4 reads.fastq       # First read in a FASTQ file (4 lines per read)

# tail -f -- Follow a file in real time (indispensable for monitoring logs)
# tail -f alignment.log     # Watch a log file as it grows

# wc -- Word/line/character count
wc -l file.txt              # Count lines
wc -c file.txt              # Count bytes

### Bioinformatics file formats you will encounter

| Format | Extension | What it contains | Typical size |
|--------|-----------|------------------|--------------|
| FASTA | `.fa`, `.fasta` | Sequences (genome, proteins) | MB to GB |
| FASTQ | `.fq`, `.fastq` | Sequencing reads + quality scores | GB (compressed) |
| SAM/BAM | `.sam`, `.bam` | Read alignments (BAM = binary compressed SAM) | GB |
| BED | `.bed` | Genomic intervals (chr, start, end) | KB to MB |
| VCF | `.vcf` | Variant calls | MB |
| GFF/GTF | `.gff`, `.gtf` | Gene annotations | MB |

**FASTA format example:**
```
>seq1 Homo sapiens BRCA1 mRNA
ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAAT
GCTATGCAGAAAATCTTAGAGTGTCCCATCTGTCTGGAGTTGATCAAGG
>seq2 Mus musculus TP53 mRNA
ATGACTGCCATGGAGGAGTCACAGTCGGATATCAGCCTCGAGCTCCCTC
```

**FASTQ format example (4 lines per read):**
```
@SEQ_ID_001
GATTTGGGGTTCAAAGCAGTATCGATCAAATAGTAAATCCATTTGTTCAACTCACAGTTT
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
```

### Text Editors in the Terminal

| Editor | Difficulty | Best for |
|--------|------------|----------|
| `nano` | Beginner | Quick edits. Ctrl+O to save, Ctrl+X to exit. |
| `vim`  | Steep learning curve | Power editing, scripting, servers. |
| `emacs`| Steep learning curve | Full IDE-like experience. |

**Vim survival guide** (you WILL encounter vim on servers):

```
vim filename          # Open file
i                     # Enter INSERT mode (now you can type)
Esc                   # Return to NORMAL mode
:w                    # Save (write)
:q                    # Quit
:wq                   # Save and quit
:q!                   # Quit without saving
/pattern              # Search forward for pattern
n / N                 # Next / previous search match
dd                    # Delete current line
u                     # Undo
Ctrl+R                # Redo
gg / G                # Go to beginning / end of file
:21                   # Go to line 21
:%s/old/new/g         # Replace all occurrences of 'old' with 'new'
```

---

## 5. Pipes and Redirections

This is the Unix superpower. Each command does one thing well; pipes (`|`) chain them
together into powerful data-processing pipelines.

```
Data flow:

  command1 | command2 | command3 > output.txt
      |         |          |          |
      v         v          v          v
  [stdout] -> [stdin] -> [stdin] -> [file]

Redirection operators:
  >    Redirect stdout to file (overwrite)
  >>   Redirect stdout to file (append)
  2>   Redirect stderr to file
  2>>  Redirect stderr to file (append)
  2>&1 Redirect stderr to same place as stdout
  <    Use file as stdin
  |    Pipe stdout of left command to stdin of right command
```

In [ ]:
%%bash
# Basic redirection
echo "Sample_001" > sample_list.txt      # Create/overwrite file
echo "Sample_002" >> sample_list.txt     # Append to file

# Separate stdout and stderr
# command > output.log 2> error.log

# Combine stdout and stderr into one file
# command > all_output.log 2>&1

# Pipe chains -- the bread and butter of command-line bioinformatics
# Count unique genes in a list:
# cat genes.txt | sort | uniq | wc -l

# Count reads in a FASTQ file (4 lines per read):
# cat sample.fastq | wc -l | awk '{print $1/4}'

# Extract sequence IDs from FASTA, clean them up, save to file:
# grep "^>" sequences.fasta | cut -d' ' -f1 | sed 's/>//' > sequence_ids.txt

### Bioinformatics pipeline examples

```bash
# Count variants per chromosome from a VCF file
grep -v "^#" variants.vcf | cut -f1 | sort | uniq -c | sort -rn

# Find unique chromosomes in a BED file, sorted by frequency
cut -f1 regions.bed | sort | uniq -c | sort -rn

# Estimate GC content of a genome
grep -v "^>" genome.fa | tr -d '\n' | \
  awk '{gc=gsub(/[GC]/,"",$0); at=gsub(/[AT]/,"",$0); print gc/(gc+at)*100"%"}'

# Count reads in a compressed FASTQ
zcat sample.fastq.gz | wc -l | awk '{print $1/4, "reads"}'
```

---

## 6. Searching: grep and find

### grep -- Search Within Files

`grep` is your go-to tool for finding patterns in text. The name stands for
"Global Regular Expression Print."

In [ ]:
%%bash
# Basic grep usage
# grep "pattern" filename

# Useful flags:
# grep -c "pattern" file      # Count matching lines (not matches)
# grep -i "pattern" file      # Case-insensitive
# grep -v "pattern" file      # Invert: show lines that do NOT match
# grep -r "pattern" dir/      # Recursive search through all files in directory
# grep -n "pattern" file      # Show line numbers
# grep -w "gene" file         # Match whole word only (not "genes" or "genebank")
# grep -l "pattern" *.txt     # List filenames that contain the pattern
# grep -A 3 "pattern" file    # Show 3 lines After each match
# grep -B 2 "pattern" file    # Show 2 lines Before each match

# Search compressed files without decompressing
# zgrep "PASS" variants.vcf.gz

#### grep in bioinformatics

```bash
# Count sequences in a FASTA file
grep -c "^>" proteins.fasta

# Extract header lines from FASTA
grep "^>" proteins.fasta

# Find all lines mentioning BRCA1 in an annotation file
grep "BRCA1" gencode.gtf

# Count TATA boxes in promoter sequences
grep -c "TATAAA" promoters.fa

# Remove comment lines from a VCF to get only data rows
grep -v "^#" variants.vcf

# Find which samples contain a specific variant
grep -l "rs12345" *.vcf

# Search for a motif and show the full FASTA header above it
grep -B 1 "GAATTC" sequences.fasta    # EcoRI recognition site
```

### find -- Locate Files in the Filesystem

In [ ]:
%%bash
# find [where to look] [criteria]

# By name
# find . -name "*.fastq"                     # Find all FASTQ files
# find . -name "*.fasta" -o -name "*.fa"     # FASTA with either extension

# By size
# find . -size +1G                            # Files larger than 1 GB
# find . -size -100k                          # Files smaller than 100 KB

# By modification time
# find . -mtime -7                            # Modified in the last 7 days

# Execute a command on each result
# find . -name "*.bam" -exec samtools index {} \;
# find . -name "*_temp*" -exec rm {} \;       # Delete all temp files

# Combine with other commands
# find . -name "*.fastq.gz" | xargs du -sh    # Show sizes of all FASTQ files

---

## 7. Text Processing Power Tools: cut, sort, uniq, awk, sed

These tools are used constantly when working with tab-delimited bioinformatics formats
(BED, VCF, GTF, SAM).

### cut -- Extract Columns

```bash
# By default, cut uses tab as delimiter
cut -f1,3 genes.bed              # Columns 1 and 3 from a BED file
cut -f1-4 data.tsv               # Columns 1 through 4
cut -d',' -f2 data.csv           # Use comma as delimiter, extract column 2
```

### sort -- Sort Lines

```bash
sort file.txt                    # Alphabetical sort
sort -n file.txt                 # Numeric sort
sort -k2,2n file.txt             # Sort by column 2, numerically
sort -k1,1 -k2,2n file.bed      # Sort by column 1 (alpha), then column 2 (numeric)
sort -rn file.txt                # Reverse numeric sort (largest first)
sort -u file.txt                 # Sort and remove duplicates
```

### uniq -- Deduplicate (requires sorted input)

```bash
sort genes.txt | uniq            # Remove duplicate lines
sort genes.txt | uniq -c         # Count occurrences of each unique line
sort genes.txt | uniq -d         # Show only duplicate lines
```

### awk -- Programmable Text Processing

`awk` processes text line by line, splitting each line into fields (`$1`, `$2`, etc.).
It is incredibly powerful for tabular data.

```
Basic syntax:   awk 'PATTERN { ACTION }' file

Special variables:
  $0     Entire line
  $1     First field
  $NF    Last field
  NR     Current line number
  NF     Number of fields in current line
  FS     Field separator (default: whitespace)
  OFS    Output field separator
```

In [ ]:
%%bash
# awk examples with bioinformatics context

# Print columns 1 and 4 from a BED file
# awk '{print $1, $4}' genes.bed

# Filter: only rows where feature length > 1000
# awk '($3 - $2) > 1000' regions.bed

# Sum a column (e.g., read counts)
# awk '{sum += $4} END {print "Total:", sum}' counts.bed

# Calculate average of column 5
# awk '{sum += $5; n++} END {print "Average:", sum/n}' data.tsv

# Use tab as delimiter explicitly (BED, VCF, GTF files are tab-delimited)
# awk -F'\t' '{print $1, $4, $5}' annotations.gtf

# Print line numbers alongside data (useful for debugging)
# awk '{print NR, $0}' file.txt

# Convert BED to a simple region string (chr:start-end)
# awk -F'\t' '{print $1 ":" $2 "-" $3}' regions.bed

echo 'awk examples -- uncomment and run with real files'

### sed -- Stream Editor

`sed` performs text transformations (substitution, deletion, insertion) on streams of text.

```bash
# Substitute first occurrence per line
sed 's/old/new/' file.txt

# Substitute ALL occurrences per line (g = global)
sed 's/old/new/g' file.txt

# Delete lines matching a pattern
sed '/^#/d' file.vcf              # Remove comment lines from VCF

# Print only lines 10-20
sed -n '10,20p' file.txt

# In-place editing (modifies the file directly)
sed -i 's/chr//' regions.bed      # Remove 'chr' prefix from chromosome names

# Chain multiple operations
sed -e 's/old/new/g' -e '/^$/d' file.txt   # Replace AND remove empty lines
```

### Bioinformatics text-processing recipes

These one-liners come up constantly in daily bioinformatics work:

```bash
# Convert FASTQ to FASTA
sed -n '1~4s/^@/>/p;2~4p' reads.fastq > reads.fasta

# Extract gene names from a GTF file
awk -F'\t' '$3=="gene"' gencode.gtf | grep -o 'gene_name "[^"]*"' | sed 's/gene_name "//;s/"//' | sort -u

# Calculate average read length from FASTQ
awk 'NR%4==2 {sum+=length($0); count++} END {print sum/count}' reads.fastq

# Get chromosome sizes from a FASTA index
cut -f1,2 genome.fa.fai

# Count reads per chromosome from SAM (skip header)
grep -v "^@" aligned.sam | cut -f3 | sort | uniq -c | sort -rn

# Filter BED file to keep only entries on chromosome 1
awk '$1=="chr1"' regions.bed
```

---

## 8. Wildcards and Pattern Matching (Globbing)

Wildcards let you refer to multiple files at once without typing each name.

In [ ]:
%%bash
# *     Match any number of characters
# ?     Match exactly one character
# [abc] Match any one of a, b, or c
# [0-9] Match any single digit

# Examples:
# ls *.fastq.gz                  # All FASTQ.GZ files
# ls sample_?.bam                # sample_1.bam, sample_2.bam, etc.
# ls sample_[123].bam            # Only samples 1, 2, 3
# ls chr[0-9]*.fa                # chr1.fa, chr22.fa, etc.
# rm *_temp*                     # Delete all temp files

echo 'Wildcard examples -- run with actual files in your directory'

---

## 9. File Permissions

Linux controls who can read, write, or execute each file. This matters when:
- You write a script and need to make it executable
- You share data on a multi-user server
- A tool refuses to run because it lacks execute permission

```
ls -l output:
  -rwxr-xr--  1  user  group  4096  Jan 15  script.sh
  |||||||||||
  |+--+--+--+
  |  |   |  |
  |  |   |  +-- Others: r-- (read only)
  |  |   +----- Group:  r-x (read + execute)
  |  +--------- Owner:  rwx (read + write + execute)
  +------------ Type:   - (regular file), d (directory)

  r = read (4)    w = write (2)    x = execute (1)
```

In [ ]:
%%bash
# chmod -- Change permissions
# chmod +x script.sh              # Add execute permission (most common use)
# chmod 755 script.sh             # rwxr-xr-x (owner: all, group/others: read+execute)
# chmod 644 data.txt              # rw-r--r-- (owner: read+write, others: read only)

# Numeric codes:
#   7 = rwx (4+2+1)    6 = rw- (4+2)    5 = r-x (4+1)    4 = r-- (4)

# chown -- Change ownership (requires sudo)
# sudo chown user:group file.txt

echo 'Permission examples'

---

## 10. File Compression and Archives

Genomics data is huge. A single human genome as a FASTQ file is ~100 GB uncompressed.
Compression is not optional -- it is standard practice.

| Format | Commands | Typical use |
|--------|----------|-------------|
| `.gz` | `gzip` / `gunzip` | FASTQ, VCF, FASTA -- the default |
| `.bz2` | `bzip2` / `bunzip2` | Better compression ratio, slower |
| `.tar` | `tar` | Archive (bundle files, no compression) |
| `.tar.gz` | `tar -czvf` / `tar -xzvf` | Archive + compression |
| `.zip` | `zip` / `unzip` | Cross-platform archives |

In [ ]:
%%bash
# gzip / gunzip -- the standard for sequencing data
# gzip large_file.fastq              # Compress (removes original by default)
# gzip -k large_file.fastq           # Compress but keep original (-k)
# gunzip file.fastq.gz               # Decompress

# View compressed files WITHOUT decompressing (saves time and disk space)
# zcat file.fastq.gz | head -4       # First read
# zgrep "^@" file.fastq.gz | wc -l   # Count reads (approximate)

# tar -- create/extract archives
# tar -czvf archive.tar.gz directory/     # Create compressed archive
#   c = create, z = gzip, v = verbose, f = filename
# tar -xzvf archive.tar.gz               # Extract
#   x = extract
# tar -tvf archive.tar.gz                # List contents without extracting
#   t = list

# bzip2 -- higher compression, slower (good for archival)
# bzip2 file.txt                     # Compress
# bunzip2 file.txt.bz2              # Decompress
# tar -cjvf archive.tar.bz2 dir/    # j = bzip2 (instead of z = gzip)

echo 'Compression examples -- uncomment to use with real files'

---

## 11. Downloading Files

You will frequently download reference genomes, annotation files, and datasets from
public databases.

In [ ]:
%%bash
# wget -- download files from URLs
# wget https://url.to/reference_genome.fa.gz
# wget -P /data/references/ URL          # Download to specific directory (-P)
# wget -O custom_name.fa.gz URL          # Download with custom filename (-O)
# wget -c URL                            # Continue interrupted download (-c)
# wget --spider URL                      # Check if URL is valid without downloading
# wget -i urls.txt                       # Download all URLs listed in a file
# wget -r -l 2 -A "*.fastq.gz" URL      # Recursive download, depth 2, only FASTQ files

# curl -- more flexible alternative (also handles APIs)
# curl -O URL                            # Download file
# curl -L URL                            # Follow redirects

echo 'Download examples -- uncomment to use'

---

## 12. Remote Server Operations (SSH and SCP)

Most serious bioinformatics work happens on remote servers or HPC clusters. You
connect via SSH (Secure Shell).

In [ ]:
%%bash
# SSH -- connect to a remote server
# ssh username@server.university.edu
# ssh -p 2222 user@server              # Specify port

# SCP -- copy files between local and remote
# scp local_file.txt user@server:~/remote_dir/          # Upload
# scp user@server:~/data/results.csv ./                 # Download
# scp -r local_dir/ user@server:~/                      # Upload directory
# scp -P 2222 file.txt user@server:~/                   # Specify port

# rsync -- smarter copy (only transfers changes, resumes interruptions)
# rsync -avz local_dir/ user@server:~/remote_dir/       # Sync local to remote
#   a = archive mode, v = verbose, z = compress during transfer

echo 'SSH/SCP examples -- use with your actual server credentials'

### tmux -- Terminal Multiplexer (essential for servers)

When your SSH connection drops, any running process dies. `tmux` keeps processes
alive in detachable sessions.

```bash
tmux new -s alignment          # Start a named session
# ... start your long-running job ...
# Ctrl+B, then D               # Detach (job keeps running)

# Later (even from a different SSH connection):
tmux attach -t alignment       # Reattach to the session
tmux ls                        # List all sessions

# Inside tmux (Ctrl+B is the prefix key):
# Ctrl+B c       New window
# Ctrl+B n / p   Next / Previous window
# Ctrl+B 0-9     Jump to window by number
# Ctrl+B %       Split pane horizontally
# Ctrl+B "       Split pane vertically
# Ctrl+B d       Detach from session
# Ctrl+B PgUp    Scroll through history
# exit           Close current pane/window
```

---

## 13. Process Management

When running long bioinformatics jobs, you need to know how to manage processes.

In [ ]:
%%bash
# Running processes in the background
# long_command &                  # Run in background (& at the end)

# Keyboard shortcuts (in the terminal, not in Jupyter)
# Ctrl+C                          # Kill the current process
# Ctrl+Z                          # Suspend (pause) the current process
# fg                              # Resume suspended process in foreground
# bg                              # Resume suspended process in background
# jobs                            # List background/suspended jobs
# fg %1                           # Bring job 1 to foreground
# bg %2                           # Resume job 2 in background

# Viewing processes
# ps aux                          # All running processes
# ps aux | grep bowtie2           # Find a specific process
# top                             # Interactive process viewer
# top -u username                 # Show only your processes
# htop                            # Better interactive viewer (install separately)
# kill PID                        # Kill a process by its PID
# kill -9 PID                     # Force kill (last resort)

# System information
# free -h                         # RAM usage (human-readable)
# free -g                         # RAM usage in GB
# nproc                           # Number of CPU cores
# lscpu                           # Detailed CPU info
# df -h                           # Disk space usage
# du -sh directory/               # Size of a specific directory

echo 'Process management examples'

---

## 14. Installing Software

On your own Linux machine or when you have admin access:

```bash
# Debian/Ubuntu package manager (apt)
sudo apt-get update               # Update package lists
sudo apt-get install samtools      # Install a package
sudo apt-get remove samtools       # Remove a package
sudo apt-get upgrade               # Upgrade all installed packages
```

For bioinformatics, **conda** (via Miniconda or Mamba) is the preferred way to install
tools because it handles complex dependencies and does not require admin access:

```bash
# Install conda (one time)
# wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
# bash Miniconda3-latest-Linux-x86_64.sh

# Create an environment for your project
conda create -n rnaseq python=3.10 fastqc trimmomatic hisat2 samtools
conda activate rnaseq

# Install from bioconda channel
conda install -c bioconda bwa star salmon
```

---

## 15. Working with BAM Files from the Command Line

BAM files (Binary Alignment Map) store read alignments. They are binary, so you
cannot `cat` or `grep` them directly. Use `samtools` instead.

```bash
# View BAM as human-readable SAM
samtools view aligned.bam | head -5

# View header only
samtools view -H aligned.bam

# Count aligned reads
samtools view -c -F 4 aligned.bam       # -F 4 = exclude unmapped

# Index a BAM file (required for many tools)
samtools index aligned.bam

# Extract reads from a specific region
samtools view aligned.bam chr17:7571720-7590868   # TP53 region

# Get basic alignment statistics
samtools flagstat aligned.bam

# Sort a BAM file (required before indexing)
samtools sort -@ 4 -o sorted.bam input.bam
```

---

## Quick Reference Card

```
NAVIGATION                  FILES                      VIEWING
  pwd                         mkdir -p dir/sub           cat file
  ls -lah                     touch file                 less file
  cd path                     cp -r src dst              head -n 10 file
  cd ..                       mv old new                 tail -f file
  cd ~                        rm -r dir                  wc -l file

SEARCHING                   TEXT PROCESSING            COMPRESSION
  grep pattern file           cut -f1,3 file.bed         gzip file
  grep -r pattern dir/        sort / uniq                gunzip file.gz
  find . -name "*.fa"         awk '{print $1}' file      tar -czvf a.tar.gz dir/
  grep -v "^#" file           sed 's/old/new/g' file     tar -xzvf a.tar.gz

PIPES & REDIRECTION         REMOTE                     PROCESSES
  cmd > file                  ssh user@server            command &
  cmd >> file                 scp file user@server:~/    Ctrl+C / Ctrl+Z
  cmd 2> errlog               tmux new -s name           fg / bg / jobs
  cmd1 | cmd2                 tmux attach -t name        ps aux / top / kill
```

---

## Exercises

Work through these exercises to practice the commands. Each exercise builds on
realistic bioinformatics scenarios.

### Exercise 1: Project Setup and File Organization

Create a complete project directory structure for a ChIP-seq analysis project.
The structure should have:
- `chipseq_project/` as the root
- Subdirectories: `data/raw`, `data/processed`, `results/peaks`, `results/plots`, `scripts`, `logs`
- An empty `README.md` in the root
- Empty Ready files: `scripts/run_pipeline.sh` and `scripts/peak_calling.sh`
- Make both `.sh` files executable

Do everything in as few commands as possible.

In [ ]:
%%bash
# YOUR SOLUTION:


### Exercise 2: FASTA File Manipulation

Given the sample FASTA file created below, write commands to:
1. Count the number of sequences
2. Extract just the header lines (without the `>` symbol)
3. Find which sequences contain the motif `GAATTC` (EcoRI restriction site)
4. Calculate the total number of nucleotide characters (excluding headers and newlines)

In [ ]:
%%bash
# Create a sample FASTA file
cat > /tmp/sample_sequences.fasta << 'EOF'
>seq1 Homo sapiens BRCA1
ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAAA
ATCTTAGAGTGTCCCATCTGAATTCGGAGTTGATCAAGGAACCTGTCTCCACAAAGTGTG
>seq2 Mus musculus TP53
ATGACTGCCATGGAGGAGTCACAGTCGGATATCAGCCTCGAGCTCCCTCTGAGCCAGGAG
ACATTTTCAGGCTTATGGAAACTACTTCCTCCAGAAGATATCCTGCCATCACCTCACTGCA
>seq3 Drosophila melanogaster Notch
ATGCAGCCGCCGCCGCCACCGCCACCGCCGCTGCTGCTGCTGCTACTGCTATTGGGATCG
GAATTCATGCAGTGCGAGTGCGAGTGCCAGTGCGAGTGCGGATCCAATGGCAATGGCAA
>seq4 Saccharomyces cerevisiae GAL4
ATGAAGCTACTGTCTTCTATCGAACAAGCATGCGATATTTGCCGACTTAAAAAGCTCAAG
TGCTCCAAAGAAAAACCGAAGTGCGCCAAGTGTCTGAAGAACAACTGGGAGTGTCGCTAC
EOF

# YOUR SOLUTIONS:
echo "=== 1. Number of sequences ==="

echo "=== 2. Headers without > ==="

echo "=== 3. Sequences containing GAATTC ==="

echo "=== 4. Total nucleotides ==="


### Exercise 3: VCF Processing Pipeline

Given the sample VCF below, write a single pipeline (using pipes) that:
1. Removes all header/comment lines (starting with `#`)
2. Extracts the chromosome (column 1) and quality score (column 6)
3. Filters to keep only rows with quality > 30
4. Sorts by quality (highest first)
5. Saves the result to `/tmp/high_quality_variants.txt`

In [ ]:
%%bash
# Create a sample VCF file
cat > /tmp/sample.vcf << 'EOF'
##fileformat=VCFv4.2
##source=GATK
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
chr1	12345	rs001	A	G	45.5	PASS	DP=30
chr1	67890	rs002	C	T	12.3	LowQual	DP=5
chr2	11111	rs003	G	A	99.0	PASS	DP=50
chr2	22222	rs004	T	C	28.7	LowQual	DP=10
chr3	33333	rs005	A	T	67.2	PASS	DP=40
chr3	44444	rs006	G	C	5.1	LowQual	DP=3
chrX	55555	rs007	C	A	88.8	PASS	DP=45
EOF

# YOUR SOLUTION:


### Exercise 4: Text Processing with awk and sed

Given the BED file below:
1. Use `awk` to add a new column that is the feature length (end - start)
2. Use `awk` to calculate the average feature length
3. Use `sed` to remove the `chr` prefix from all chromosome names
4. Use `awk` to convert the BED entries into the format `chr:start-end`

In [ ]:
%%bash
# Create a sample BED file
cat > /tmp/sample.bed << 'EOF'
chr1	1000	5000	gene_A	100	+
chr1	8000	9500	gene_B	200	-
chr2	15000	25000	gene_C	300	+
chr3	500	800	gene_D	50	+
chrX	40000	42000	gene_E	150	-
EOF

echo "=== 1. BED with feature length column ==="
# YOUR SOLUTION:

echo "=== 2. Average feature length ==="
# YOUR SOLUTION:

echo "=== 3. Remove chr prefix ==="
# YOUR SOLUTION:

echo "=== 4. Convert to chr:start-end format ==="
# YOUR SOLUTION:


### Exercise 5: Putting It All Together

You have received sequencing data. Write the commands to:
1. Create a project directory structure with folders: `raw`, `qc`, `aligned`, `results`
2. Simulate downloading a reference genome (just create a dummy file)
3. Write a one-liner that finds all `.fastq.gz` files in a directory, counts the number of reads in each (4 lines per read in FASTQ), and reports filename and read count
4. Show how you would compress the `results/` directory into an archive for sharing with a collaborator

In [ ]:
%%bash
# YOUR SOLUTION:


---

## Exercise Solutions

### Solution 1

In [ ]:
%%bash
# Create full project structure in one command
mkdir -p chipseq_project/{data/{raw,processed},results/{peaks,plots},scripts,logs}

# Create files
touch chipseq_project/README.md
touch chipseq_project/scripts/{run_pipeline.sh,peak_calling.sh}

# Make scripts executable
chmod +x chipseq_project/scripts/*.sh

# Verify
find chipseq_project -type f | sort
ls -la chipseq_project/scripts/

### Solution 2

In [ ]:
%%bash
echo "=== 1. Number of sequences ==="
grep -c "^>" /tmp/sample_sequences.fasta

echo "=== 2. Headers without > ==="
grep "^>" /tmp/sample_sequences.fasta | sed 's/^>//' 

echo "=== 3. Sequences containing GAATTC ==="
# Use grep -B 1 to also show the header line above the match
grep -B 1 "GAATTC" /tmp/sample_sequences.fasta | grep "^>"

echo "=== 4. Total nucleotides ==="
grep -v "^>" /tmp/sample_sequences.fasta | tr -d '\n' | wc -c

### Solution 3

In [ ]:
%%bash
grep -v "^#" /tmp/sample.vcf | \
  awk -F'\t' '{print $1, $6}' | \
  awk '$2 > 30' | \
  sort -k2,2rn > /tmp/high_quality_variants.txt

echo "Result:"
cat /tmp/high_quality_variants.txt

### Solution 4

In [ ]:
%%bash
echo "=== 1. BED with feature length column ==="
awk -F'\t' '{print $0 "\t" $3-$2}' /tmp/sample.bed

echo ""
echo "=== 2. Average feature length ==="
awk -F'\t' '{len=$3-$2; sum+=len; n++} END {print sum/n}' /tmp/sample.bed

echo ""
echo "=== 3. Remove chr prefix ==="
sed 's/^chr//' /tmp/sample.bed

echo ""
echo "=== 4. Convert to chr:start-end format ==="
awk -F'\t' '{print $1 ":" $2 "-" $3}' /tmp/sample.bed

### Solution 5

In [ ]:
%%bash
# 1. Create project structure
mkdir -p /tmp/seq_project/{raw,qc,aligned,results}

# 2. Simulate downloading a reference (create a dummy FASTA)
echo -e ">chr1\nACGTACGTACGT" > /tmp/seq_project/reference.fa

# 3. Create dummy FASTQ files for testing, then count reads
# (A FASTQ read is 4 lines: @header, sequence, +, quality)
for i in 1 2 3; do
    printf '@read1\nACGT\n+\nIIII\n@read2\nGCTA\n+\nIIII\n' | gzip > /tmp/seq_project/raw/sample_${i}.fastq.gz
done

echo "Read counts per file:"
for f in /tmp/seq_project/raw/*.fastq.gz; do
    reads=$(zcat "$f" | wc -l | awk '{print $1/4}')
    echo "  $(basename $f): $reads reads"
done

# 4. Archive results for sharing
# tar -czvf /tmp/seq_project_results.tar.gz -C /tmp/seq_project results/
echo ""
echo "Archive command: tar -czvf results_archive.tar.gz results/"

---

## Key Takeaways

1. **Navigation**: `pwd`, `ls -lah`, and `cd` will be your most-typed commands
2. **Pipes are the Unix superpower**: `grep | cut | sort | uniq` chains handle 80% of text processing
3. **grep is your best friend**: Learn it well for searching FASTA, VCF, GTF, and log files
4. **awk for columns, sed for substitutions**: These two handle the rest of the 20%
5. **Always use compression**: `gzip` for individual files, `tar.gz` for directories
6. **tmux on servers**: Never run long jobs without tmux -- you WILL lose SSH connections
7. **Be careful with `rm -rf`**: There is no undo. Double-check before pressing Enter.

---

### Next Module: Git Version Control -->

---

[< Previous: Tier 0 Skills Check: Can You Skip Computational...](../00_Skills_Check/00_skills_check.ipynb) | [Tier 0: Computational Foundations Overview](../README.md) | [Next: Module 0.2: Git Version Control for Scientists >](../02_Git_Version_Control/01_git_version_control.ipynb)

---